In [1]:
import numpy as np
import pandas as pd
from haversine import haversine

# 1. 청룡동

In [2]:
입지후보지 = pd.read_csv('입지후보지.csv')
입지후보지 = 입지후보지[입지후보지['행정동'] == '청룡동']
후보지_points = np.array([list(i) for i in zip(입지후보지['위도'], 입지후보지['경도'])])

버스 = pd.read_csv('청룡동_버스정류장.csv')
버스_points = np.array([list(i) for i in zip(버스['Y좌표'], 버스['X좌표'])])

노인시설 = pd.read_csv('청룡동_노인시설.csv', index_col=0)
노인시설_points = np.array([list(i) for i in zip(노인시설['위도'], 노인시설['경도'])])

노인복지시설 = pd.read_csv('청룡동_노인복지시설.csv', index_col=0)
노인복지시설_points = np.array([list(i) for i in zip(노인복지시설['위도'], 노인복지시설['경도'])])

경로당 = pd.read_csv('청룡동_경로당.csv', index_col=0)
경로당_points = np.array([list(i) for i in zip(경로당['위도'], 경로당['경도'])])
경로당 = 경로당.reset_index()

In [3]:
X = list(노인시설['위도']) + list(노인복지시설['위도']) + list(경로당['위도']) + list(버스['Y좌표'])
Y = list(노인시설['경도']) + list(노인복지시설['경도']) + list(경로당['경도']) + list(버스['X좌표'])
points = np.array([list(i) for i in zip(X, Y)])
print(points.shape)
points[:2]

(138, 2)


array([[ 36.80013535, 127.1776807 ],
       [ 36.76749349, 127.1650992 ]])

In [4]:
전체w = points.shape[0]
버스w = 버스.shape[0]
노인시설w = 노인시설.shape[0]
노인복지시설w = 노인복지시설.shape[0]
경로당w = 경로당.shape[0]

# 가중치는 (전체수요지점수 - 해당수요지점수) / 전체수요지점수

m1 = (전체w-버스w) / 전체w
m2 = (전체w-노인시설w) / 전체w
m3 = (전체w-노인복지시설w) / 전체w
m4 = (전체w-경로당w) / 전체w

In [5]:
def pmedian(후보지_points, 버스_points, 노인시설_points, 노인복지시설_points, 경로당_points):
    
    ### 버스
    havers1 = []
    for i in 후보지_points:
        site1 = []
        for j in 버스_points:
            site1.append(haversine(i,j)*1000)
        havers1.append(site1)

    location = list(입지후보지['이름'])
    location1 = list(버스['정류장명'])

    havers_D1 = dict(zip(location, [dict(zip(location1, i)) for i in havers1]))
    D1 = pd.DataFrame(havers_D1)
    
    최소값 = list(D1.min(axis=1))

    # 최단거리인 경우 가중치 부여, 아니면 0
    idx = D1.index
    col = D1.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D1.loc[idx[i], col[j]] == 최소값[i]:
                D1.loc[idx[i], col[j]] = m1
            else:
                D1.loc[idx[i], col[j]] = 0
    
    
     ### 노인시설
    havers2 = []
    for i in 후보지_points:
        site2 = []
        for j in 노인시설_points:
            site2.append(haversine(i,j)*1000)
        havers2.append(site2)

    location2 = list(노인시설['시설명'])

    havers_D2 = dict(zip(location, [dict(zip(location2, i)) for i in havers2]))
    D2 = pd.DataFrame(havers_D2)
    
    최소값 = list(D2.min(axis=1))

    idx = D2.index
    col = D2.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D2.loc[idx[i], col[j]] == 최소값[i]:
                D2.loc[idx[i], col[j]] = m2
            else:
                D2.loc[idx[i], col[j]] = 0


    ### 노인복지시설설
    havers3 = []
    for i in 후보지_points:
        site3 = []
        for j in 노인복지시설_points:
            site3.append(haversine(i,j)*1000)
        havers3.append(site3)

    location3 = list(노인복지시설['이름'])

    havers_D3 = dict(zip(location, [dict(zip(location3, i)) for i in havers3]))
    D3 = pd.DataFrame(havers_D3)
    
    최소값 = list(D3.min(axis=1))

    idx = D3.index
    col = D3.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D3.loc[idx[i], col[j]] == 최소값[i]:
                D3.loc[idx[i], col[j]] = m3
            else:
                D3.loc[idx[i], col[j]] = 0
             
                 
    ### 경로당
    havers4 = []
    for i in 후보지_points:
        site4 = []
        for j in 경로당_points:
            site4.append(haversine(i,j)*1000)
        havers4.append(site4)

    location4 = list(경로당['경로당명'])

    havers_D4 = dict(zip(location, [dict(zip(location4, i)) for i in havers4]))
    D4 = pd.DataFrame(havers_D4)
    
    최소값 = list(D4.min(axis=1))

    idx = D4.index
    col = D4.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D4.loc[idx[i], col[j]] == 최소값[i]:
                D4.loc[idx[i], col[j]] = m4
            else:
                D4.loc[idx[i], col[j]] = 0
                
    D_final = pd.concat([D1, D2, D3, D4])
    print(D_final.sum().sort_values(ascending=False))

    # 최종 후보지 저장
    results = D_final.sum().sort_values(ascending=False)
    최종후보지 = 입지후보지[입지후보지['이름'].isin(results.head(5).index.tolist())].copy()
    최종후보지 = 최종후보지[['이름', '위도', '경도']]

    최종후보지.to_csv('청룡동_후보지(P-median).csv', index=False, encoding='cp949')

In [6]:
pmedian(후보지_points, 버스_points, 노인시설_points, 노인복지시설_points, 경로당_points)

삼룡3통 경로당                 4.463768
구성5통 경로당                 3.507246
청당코오롱하늘채                 3.253623
구룡1통 경로당                 3.253623
청룡11통 경로당                2.166667
신도브래뉴 경로당                2.101449
한양수자인 경로당                1.594203
삼룡1통 경로당                 1.463768
부영사랑으로 경로당               1.340580
청당우미린 경로당                1.340580
극동1차 경로당                 1.340580
경남아너스빌 경로당               1.340580
청룡14통 경로당                1.340580
구룡3통 마을회관 앞 정자           1.086957
청당산운 경로당                 1.086957
중흥S클래스 경로당               1.086957
한양수자인블루시티(아)             1.086957
꿈에그린 경로당                 1.086957
향목그라 경로당                 1.086957
청당1공원                    1.086957
행정타운 두산위브(아) 경로당         1.086957
LGSK 경로당                 0.833333
청수호수공원                   0.833333
구광 경로당                   0.833333
구성휴먼시아 경로당               0.833333
뒷들공원(청당7공원)              0.760870
삼룡2통 경로당                 0.507246
구룡2리경로당                  0.507246
하천변                      0.507246
청룡동 주민센터      

# 2. 성환읍

In [7]:
입지후보지 = pd.read_csv('입지후보지.csv')
입지후보지 = 입지후보지[입지후보지['행정동'] == '성환읍']
후보지_points = np.array([list(i) for i in zip(입지후보지['위도'], 입지후보지['경도'])])

버스 = pd.read_csv('성환읍_버스정류장.csv')
버스_points = np.array([list(i) for i in zip(버스['Y좌표'], 버스['X좌표'])])

노인시설 = pd.read_csv('성환읍_노인시설.csv', index_col=0)
노인시설_points = np.array([list(i) for i in zip(노인시설['위도'], 노인시설['경도'])])

노인복지시설 = pd.read_csv('성환읍_노인복지시설.csv', index_col=0)
노인복지시설_points = np.array([list(i) for i in zip(노인복지시설['위도'], 노인복지시설['경도'])])

경로당 = pd.read_csv('성환읍_경로당.csv', index_col=0)
경로당_points = np.array([list(i) for i in zip(경로당['위도'], 경로당['경도'])])
경로당 = 경로당.reset_index()

In [8]:
X = list(노인시설['위도']) + list(노인복지시설['위도']) + list(경로당['위도']) + list(버스['Y좌표'])
Y = list(노인시설['경도']) + list(노인복지시설['경도']) + list(경로당['경도']) + list(버스['X좌표'])
points = np.array([list(i) for i in zip(X, Y)])
print(points.shape)
points[:2]

(256, 2)


array([[ 36.94717367, 127.1513989 ],
       [ 36.92117221, 127.1374003 ]])

In [9]:
전체w = points.shape[0]
버스w = 버스.shape[0]
노인시설w = 노인시설.shape[0]
노인복지시설w = 노인복지시설.shape[0]
경로당w = 경로당.shape[0]

# 가중치는 (전체수요지점수 - 해당수요지점수) / 전체수요지점수

m1 = (전체w-버스w) / 전체w
m2 = (전체w-노인시설w) / 전체w
m3 = (전체w-노인복지시설w) / 전체w
m4 = (전체w-경로당w) / 전체w

In [10]:
def pmedian(후보지_points, 버스_points, 노인시설_points, 노인복지시설_points, 경로당_points):
    
    ### 버스
    havers1 = []
    for i in 후보지_points:
        site1 = []
        for j in 버스_points:
            site1.append(haversine(i,j)*1000)
        havers1.append(site1)

    location = list(입지후보지['이름'])
    location1 = list(버스['정류장명'])

    havers_D1 = dict(zip(location, [dict(zip(location1, i)) for i in havers1]))
    D1 = pd.DataFrame(havers_D1)
    
    최소값 = list(D1.min(axis=1))

    # 최단거리인 경우 가중치 부여, 아니면 0
    idx = D1.index
    col = D1.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D1.loc[idx[i], col[j]] == 최소값[i]:
                D1.loc[idx[i], col[j]] = m1
            else:
                D1.loc[idx[i], col[j]] = 0
    
    
     ### 노인시설
    havers2 = []
    for i in 후보지_points:
        site2 = []
        for j in 노인시설_points:
            site2.append(haversine(i,j)*1000)
        havers2.append(site2)

    location2 = list(노인시설['시설명'])

    havers_D2 = dict(zip(location, [dict(zip(location2, i)) for i in havers2]))
    D2 = pd.DataFrame(havers_D2)
    
    최소값 = list(D2.min(axis=1))

    idx = D2.index
    col = D2.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D2.loc[idx[i], col[j]] == 최소값[i]:
                D2.loc[idx[i], col[j]] = m2
            else:
                D2.loc[idx[i], col[j]] = 0


    ### 노인복지시설설
    havers3 = []
    for i in 후보지_points:
        site3 = []
        for j in 노인복지시설_points:
            site3.append(haversine(i,j)*1000)
        havers3.append(site3)

    location3 = list(노인복지시설['이름'])

    havers_D3 = dict(zip(location, [dict(zip(location3, i)) for i in havers3]))
    D3 = pd.DataFrame(havers_D3)
    
    최소값 = list(D3.min(axis=1))

    idx = D3.index
    col = D3.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D3.loc[idx[i], col[j]] == 최소값[i]:
                D3.loc[idx[i], col[j]] = m3
            else:
                D3.loc[idx[i], col[j]] = 0
             
                 
    ### 경로당
    havers4 = []
    for i in 후보지_points:
        site4 = []
        for j in 경로당_points:
            site4.append(haversine(i,j)*1000)
        havers4.append(site4)

    location4 = list(경로당['경로당명'])

    havers_D4 = dict(zip(location, [dict(zip(location4, i)) for i in havers4]))
    D4 = pd.DataFrame(havers_D4)
    
    최소값 = list(D4.min(axis=1))

    idx = D4.index
    col = D4.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D4.loc[idx[i], col[j]] == 최소값[i]:
                D4.loc[idx[i], col[j]] = m4
            else:
                D4.loc[idx[i], col[j]] = 0
                
    D_final = pd.concat([D1, D2, D3, D4])
    print(D_final.sum().sort_values(ascending=False))

    # 최종 후보지 저장
    results = D_final.sum().sort_values(ascending=False)
    최종후보지 = 입지후보지[입지후보지['이름'].isin(results.head(5).index.tolist())].copy()
    최종후보지 = 최종후보지[['이름', '위도', '경도']]

    최종후보지.to_csv('성환읍_후보지(P-median).csv', index=False, encoding='cp949')

In [11]:
pmedian(후보지_points, 버스_points, 노인시설_points, 노인복지시설_points, 경로당_points)

성월1리 경로당           6.597656
골드맨션경로당(매주4리)      5.511719
수향2길 경로당           4.312500
성환7리 경로당           3.367188
매주4리 경로당           3.355469
                     ...   
두진아파트 경로당(율금5리)    0.000000
율금6리 경로당           0.000000
마을공터               0.000000
성환8리 경로당           0.000000
성월2리 경로당           0.000000
Length: 96, dtype: float64


# 3. 신안동

In [12]:
입지후보지 = pd.read_csv('입지후보지.csv')
입지후보지 = 입지후보지[입지후보지['행정동'] == '신안동']
후보지_points = np.array([list(i) for i in zip(입지후보지['위도'], 입지후보지['경도'])])

버스 = pd.read_csv('신안동_버스정류장.csv')
버스_points = np.array([list(i) for i in zip(버스['Y좌표'], 버스['X좌표'])])

노인시설 = pd.read_csv('신안동_노인시설.csv', index_col=0)
노인시설_points = np.array([list(i) for i in zip(노인시설['위도'], 노인시설['경도'])])

노인복지시설 = pd.read_csv('신안동_노인복지시설.csv', index_col=0)
노인복지시설_points = np.array([list(i) for i in zip(노인복지시설['위도'], 노인복지시설['경도'])])

경로당 = pd.read_csv('신안동_경로당.csv', index_col=0)
경로당_points = np.array([list(i) for i in zip(경로당['위도'], 경로당['경도'])])
경로당 = 경로당.reset_index()

In [13]:
X = list(노인시설['위도']) + list(노인복지시설['위도']) + list(경로당['위도']) + list(버스['Y좌표'])
Y = list(노인시설['경도']) + list(노인복지시설['경도']) + list(경로당['경도']) + list(버스['X좌표'])
points = np.array([list(i) for i in zip(X, Y)])
print(points.shape)
points[:2]

(98, 2)


array([[ 36.81772127, 127.1525376 ],
       [ 36.81753212, 127.1518383 ]])

In [14]:
전체w = points.shape[0]
버스w = 버스.shape[0]
노인시설w = 노인시설.shape[0]
노인복지시설w = 노인복지시설.shape[0]
경로당w = 경로당.shape[0]

# 가중치는 (전체수요지점수 - 해당수요지점수) / 전체수요지점수

m1 = (전체w-버스w) / 전체w
m2 = (전체w-노인시설w) / 전체w
m3 = (전체w-노인복지시설w) / 전체w
m4 = (전체w-경로당w) / 전체w

In [15]:
def pmedian(후보지_points, 버스_points, 노인시설_points, 노인복지시설_points, 경로당_points):
    
    ### 버스
    havers1 = []
    for i in 후보지_points:
        site1 = []
        for j in 버스_points:
            site1.append(haversine(i,j)*1000)
        havers1.append(site1)

    location = list(입지후보지['이름'])
    location1 = list(버스['정류장명'])

    havers_D1 = dict(zip(location, [dict(zip(location1, i)) for i in havers1]))
    D1 = pd.DataFrame(havers_D1)
    
    최소값 = list(D1.min(axis=1))

    # 최단거리인 경우 가중치 부여, 아니면 0
    idx = D1.index
    col = D1.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D1.loc[idx[i], col[j]] == 최소값[i]:
                D1.loc[idx[i], col[j]] = m1
            else:
                D1.loc[idx[i], col[j]] = 0
    
    
     ### 노인시설
    havers2 = []
    for i in 후보지_points:
        site2 = []
        for j in 노인시설_points:
            site2.append(haversine(i,j)*1000)
        havers2.append(site2)

    location2 = list(노인시설['시설명'])

    havers_D2 = dict(zip(location, [dict(zip(location2, i)) for i in havers2]))
    D2 = pd.DataFrame(havers_D2)
    
    최소값 = list(D2.min(axis=1))

    idx = D2.index
    col = D2.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D2.loc[idx[i], col[j]] == 최소값[i]:
                D2.loc[idx[i], col[j]] = m2
            else:
                D2.loc[idx[i], col[j]] = 0


    ### 노인복지시설설
    havers3 = []
    for i in 후보지_points:
        site3 = []
        for j in 노인복지시설_points:
            site3.append(haversine(i,j)*1000)
        havers3.append(site3)

    location3 = list(노인복지시설['이름'])

    havers_D3 = dict(zip(location, [dict(zip(location3, i)) for i in havers3]))
    D3 = pd.DataFrame(havers_D3)
    
    최소값 = list(D3.min(axis=1))

    idx = D3.index
    col = D3.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D3.loc[idx[i], col[j]] == 최소값[i]:
                D3.loc[idx[i], col[j]] = m3
            else:
                D3.loc[idx[i], col[j]] = 0
             
                 
    ### 경로당
    havers4 = []
    for i in 후보지_points:
        site4 = []
        for j in 경로당_points:
            site4.append(haversine(i,j)*1000)
        havers4.append(site4)

    location4 = list(경로당['경로당명'])

    havers_D4 = dict(zip(location, [dict(zip(location4, i)) for i in havers4]))
    D4 = pd.DataFrame(havers_D4)
    
    최소값 = list(D4.min(axis=1))

    idx = D4.index
    col = D4.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D4.loc[idx[i], col[j]] == 최소값[i]:
                D4.loc[idx[i], col[j]] = m4
            else:
                D4.loc[idx[i], col[j]] = 0
                
    D_final = pd.concat([D1, D2, D3, D4])
    print(D_final.sum().sort_values(ascending=False))

    # 최종 후보지 저장
    results = D_final.sum().sort_values(ascending=False)
    최종후보지 = 입지후보지[입지후보지['이름'].isin(results.head(5).index.tolist())].copy()
    최종후보지 = 최종후보지[['이름', '위도', '경도']]

    최종후보지.to_csv('신안동_후보지(P-median).csv', index=False, encoding='cp949')

In [16]:
pmedian(후보지_points, 버스_points, 노인시설_points, 노인복지시설_points, 경로당_points)

신안중앙경로당          4.397959
신안안서대림@경로당       1.581633
신안27통경로당         1.326531
신안대림한내@경로당       1.326531
신안안서금호어울림@경로당    1.326531
신안25통경로당         1.071429
힐스테이트@경로당        1.071429
신안경남아너스빌@경로당     1.071429
마을회관             0.989796
신안24통경로당         0.816327
신안16통경로당         0.816327
신안휴먼시아@경로당       0.816327
신안동주민센터          0.816327
신안14통경로당         0.816327
신안13, 20통경로당     0.816327
신안안서부경@경로당       0.816327
신안대림한들@경로당       0.816327
신부6공원            0.816327
신부8공원            0.816327
천호지근린공원          0.765306
안서1공원            0.765306
신안동아태조@경로당       0.510204
노인회관             0.510204
마을공터             0.510204
신안26통경로당         0.510204
신안3통경로당          0.510204
신부공원             0.510204
안서2어린이공원         0.510204
신안청광플러스원@경로당     0.255102
안서3어린이공원         0.255102
신부문화공원           0.255102
신안8통경로당          0.255102
신안대림한숲@경로당       0.255102
천안향교 앞           0.000000
신부2공원            0.000000
태조산공원            0.000000
성정9공원            0.000000
천호지생활체육공원        0.000000
신부1공원       